# 03 — Fact Generation

This notebook builds the actual knowledge base of the RAG system: every aggregation computed here becomes one retrievable fact sentence in Notebook 4. Two rules throughout: **never generate a fact from fewer than 30 respondents**, and **always state the sample size in the sentence itself**, so both the LLM and the person reading the answer can judge how much to trust it.

In [1]:
import pandas as pd
from collections import Counter

DATA_DIR = "data"
df = pd.read_csv(f"{DATA_DIR}/results.txt", low_memory=False)

MIN_N = 30
facts = []

def add_fact(text, category, n):
    facts.append({"text": text, "category": category, "n": n})

## Setup

Salary cleaning and experience buckets carry over from Notebook 2. One addition here: `clean_role()`. Raw `DevType` values like `"Developer, full-stack"` read badly once you try to build a sentence around them — `"Developer, full-stacks report..."` isn't valid English. Reordering to `"full-stack developer"` fixes the common cases; everywhere a role gets used below, we also wrap it as "Among {role} respondents" rather than pluralizing it directly, so nothing breaks regardless of the exact category name.

In [2]:
df["comp_clean"] = df["ConvertedCompYearly"].where(
    (df["ConvertedCompYearly"] >= 1000) & (df["ConvertedCompYearly"] <= 2_000_000)
)

def bucket_exp(years):
    if pd.isna(years) or years > 50: return None
    if years <= 2: return "0-2 years"
    if years <= 5: return "3-5 years"
    if years <= 10: return "6-10 years"
    if years <= 20: return "11-20 years"
    return "21-50 years"
df["exp_bucket"] = df["WorkExp"].apply(bucket_exp)

def clean_role(role):
    """'Developer, full-stack' -> 'full-stack developer'; everything else unchanged."""
    if role.startswith("Developer, "):
        return role.replace("Developer, ", "", 1) + " developer"
    return role

exclude_devtypes = {"Other (please specify):", "Retired"}
devtype_counts = df["DevType"].value_counts()
devtype_allowlist = [d for d in devtype_counts.index if d not in exclude_devtypes and devtype_counts[d] >= 100]
top_countries = df["Country"].value_counts().head(15).index.tolist()
exp_buckets = ["0-2 years", "3-5 years", "6-10 years", "11-20 years", "21-50 years"]
exclude_ed = {"Other (please specify):"}

def parse_multiselect(series):
    return series.dropna().astype(str).apply(lambda x: [v.strip() for v in x.split(";") if v.strip()])

def top_n_multiselect(series, n=5):
    parsed = parse_multiselect(series)
    counts = Counter()
    for row in parsed:
        counts.update(row)
    return counts.most_common(n), len(parsed)

print(f"{len(devtype_allowlist)} roles, {len(top_countries)} countries pass the allowlist")

29 roles, 15 countries pass the allowlist


## Salary facts

Eight cuts now instead of four: role, role × country, experience, country, **education level, IC-vs-manager, company size, and remote-work status** — the first four were in the original pass, the last four fill real gaps (your own pitch deck literally asks "is my CS degree worth it?", which needs the education cut to answer at all).

In [3]:
for role in devtype_allowlist:
    r = clean_role(role)
    sub = df.loc[df["DevType"] == role, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"The median annual salary among {r} respondents is \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_role", len(sub))

for role in devtype_allowlist:
    r = clean_role(role)
    for country in top_countries:
        sub = df.loc[(df["DevType"] == role) & (df["Country"] == country), "comp_clean"].dropna()
        if len(sub) >= MIN_N:
            add_fact(f"In {country}, the median annual salary among {r} respondents is \${int(sub.median()):,}, based on {len(sub)} respondents.",
                      "salary_by_role_country", len(sub))

for bucket in exp_buckets:
    sub = df.loc[df["exp_bucket"] == bucket, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Developers with {bucket} of professional experience report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_experience", len(sub))

for country in top_countries:
    sub = df.loc[df["Country"] == country, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"The median annual salary in {country} is \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_country", len(sub))

<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:19: SyntaxWarning: invalid escape sequence '\$'
<>:25: SyntaxWarning: invalid escape sequence '\$'
<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:19: SyntaxWarning: invalid escape sequence '\$'
<>:25: SyntaxWarning: invalid escape sequence '\$'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18872\1227113490.py:5: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"The median annual salary among {r} respondents is \${int(sub.median()):,}, based on {len(sub)} respondents.",
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18872\1227113490.py:13: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"In {country}, the median annual salary among {r} respondents is \${int(sub.median()):,}, based on {len(sub)} respondents.",
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18872\1227113490.py:19: SyntaxWarning: invalid escape sequence '\$'
  

In [4]:
for ed in df["EdLevel"].dropna().unique():
    if ed in exclude_ed: continue
    sub = df.loc[df["EdLevel"] == ed, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Developers whose highest education is '{ed}' report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_education", len(sub))

for status in df["ICorPM"].dropna().unique():
    sub = df.loc[df["ICorPM"] == status, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Among {status.lower()} respondents, the median annual salary is \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_ic_or_manager", len(sub))

for size in df["OrgSize"].dropna().unique():
    if "don" in size.lower(): continue
    sub = df.loc[df["OrgSize"] == size, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Developers at companies with {size.lower()} report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_org_size", len(sub))

for status in df["RemoteWork"].dropna().unique():
    sub = df.loc[df["RemoteWork"] == status, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Developers working '{status}' report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_remote", len(sub))

print(f"Facts after salary section: {len(facts)}")

Facts after salary section: 173


<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:24: SyntaxWarning: invalid escape sequence '\$'
<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:24: SyntaxWarning: invalid escape sequence '\$'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18872\4060994970.py:5: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"Developers whose highest education is '{ed}' report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18872\4060994970.py:11: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"Among {status.lower()} respondents, the median annual salary is \${int(sub.median()):,}, based on {len(sub)} respondents.",
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18872\4060994970.py:18: SyntaxWarning: inv

## Technology adoption facts

Overall top tools (five multi-select columns), then language/database/platform broken down by role, plus top languages by country — this last one wasn't in the original pass either, and it's exactly what answers "which languages are popular in Germany?" style questions.

In [5]:
tech_columns = {
    "LanguageHaveWorkedWith": "language/technology",
    "DatabaseHaveWorkedWith": "database",
    "PlatformHaveWorkedWith": "cloud/platform tool",
    "WebframeHaveWorkedWith": "web framework",
    "AIModelsHaveWorkedWith": "AI model/LLM",
}
for col, label in tech_columns.items():
    top5, total = top_n_multiselect(df[col], n=5)
    if total >= MIN_N:
        for tech, c in top5:
            add_fact(f"{c/total*100:.0f}% of developers used {tech} as their {label} in the past year, based on {total} respondents.",
                      f"tech_adoption_overall_{col}", total)

for col, label in [("LanguageHaveWorkedWith", "language/technology"), ("DatabaseHaveWorkedWith", "database"), ("PlatformHaveWorkedWith", "cloud/platform tool")]:
    for role in devtype_allowlist:
        r = clean_role(role)
        sub_series = df.loc[df["DevType"] == role, col]
        top3, total = top_n_multiselect(sub_series, n=3)
        if total >= MIN_N:
            for tech, c in top3:
                add_fact(f"Among {r} respondents, {c/total*100:.0f}% used {tech} as their {label} in the past year, based on {total} respondents.",
                          "tech_by_role", total)

for country in top_countries:
    sub_series = df.loc[df["Country"] == country, "LanguageHaveWorkedWith"]
    top3, total = top_n_multiselect(sub_series, n=3)
    if total >= MIN_N:
        for lang, c in top3:
            add_fact(f"In {country}, {c/total*100:.0f}% of developers used {lang} in the past year, based on {total} respondents.",
                      "language_by_country", total)

print(f"Facts after technology section: {len(facts)}")

Facts after technology section: 501


## Satisfaction facts

By remote-work status and role as before, now also by education level and IC-vs-manager — relevant to "should I go into management?" type questions.

In [6]:
for status in df["RemoteWork"].dropna().unique():
    sub = df.loc[df["RemoteWork"] == status, "JobSat"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Among developers working '{status}', the average job satisfaction is {sub.mean():.1f} out of 10, based on {len(sub)} respondents.",
                  "satisfaction_by_remote", len(sub))

for role in devtype_allowlist:
    r = clean_role(role)
    sub = df.loc[df["DevType"] == role, "JobSat"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Among {r} respondents, the average job satisfaction is {sub.mean():.1f} out of 10, based on {len(sub)} respondents.",
                  "satisfaction_by_role", len(sub))

for ed in df["EdLevel"].dropna().unique():
    if ed in exclude_ed: continue
    sub = df.loc[df["EdLevel"] == ed, "JobSat"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Among developers whose highest education is '{ed}', the average job satisfaction is {sub.mean():.1f} out of 10, based on {len(sub)} respondents.",
                  "satisfaction_by_education", len(sub))

for status in df["ICorPM"].dropna().unique():
    sub = df.loc[df["ICorPM"] == status, "JobSat"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Among {status.lower()} respondents, the average job satisfaction is {sub.mean():.1f} out of 10, based on {len(sub)} respondents.",
                  "satisfaction_by_ic_or_manager", len(sub))

print(f"Facts after satisfaction section: {len(facts)}")

Facts after satisfaction section: 543


## AI perception facts

This is the section that ties most directly back to the project's actual theme, and where we had the least coverage before: sentiment by experience (already had this), plus threat perception, trust in AI accuracy, perceived ability on complex tasks, and AI agent usage — all broken down by role or experience.

In [7]:
for bucket in exp_buckets:
    sub = df.loc[df["exp_bucket"] == bucket, "AISent"].dropna()
    if len(sub) >= MIN_N:
        fav_pct = sub.isin(["Favorable", "Very favorable"]).mean() * 100
        add_fact(f"Among developers with {bucket} of experience, {fav_pct:.0f}% have a favorable view of AI tools, based on {len(sub)} respondents.",
                  "ai_sentiment_by_experience", len(sub))

for role in devtype_allowlist:
    r = clean_role(role)
    sub = df.loc[df["DevType"] == role, "AIThreat"].dropna()
    if len(sub) >= MIN_N:
        yes_pct = (sub == "Yes").mean() * 100
        add_fact(f"Among {r} respondents, {yes_pct:.0f}% see AI as a threat to their current job, based on {len(sub)} respondents.",
                  "ai_threat_by_role", len(sub))

for role in devtype_allowlist:
    r = clean_role(role)
    sub = df.loc[df["DevType"] == role, "AIAcc"].dropna()
    if len(sub) >= MIN_N:
        trust_pct = sub.isin(["Highly trust", "Somewhat trust"]).mean() * 100
        add_fact(f"Among {r} respondents, {trust_pct:.0f}% trust the accuracy of AI tool output, based on {len(sub)} respondents.",
                  "ai_trust_by_role", len(sub))

for bucket in exp_buckets:
    sub = df.loc[df["exp_bucket"] == bucket, "AIComplex"].dropna()
    if len(sub) >= MIN_N:
        good_pct = sub.isin(["Very well at handling complex tasks", "Good, but not great at handling complex tasks"]).mean() * 100
        add_fact(f"Among developers with {bucket} of experience, {good_pct:.0f}% rate AI tools as good or better at handling complex tasks, based on {len(sub)} respondents.",
                  "ai_complexity_by_experience", len(sub))

for role in devtype_allowlist:
    r = clean_role(role)
    sub = df.loc[df["DevType"] == role, "AIAgents"].dropna()
    if len(sub) >= MIN_N:
        uses_pct = sub.str.startswith("Yes").mean() * 100
        add_fact(f"Among {r} respondents, {uses_pct:.0f}% currently use AI agents in their work, based on {len(sub)} respondents.",
                  "ai_agent_usage_by_role", len(sub))

print(f"Facts after AI perception section: {len(facts)}")

Facts after AI perception section: 640


## Salary and adoption by specific language

The per-role tech facts above only capture the *top 3* languages for each role — which means a real but less-common choice like Rust or Go can end up with zero coverage anywhere, even though people specifically ask about them (your own pitch example is exactly this: junior dev deciding between Rust and Go). This section fixes that by computing a fact for every one of the top 20 languages directly, independent of role.

In [8]:
top20_langs, lang_total = top_n_multiselect(df["LanguageHaveWorkedWith"], n=20)

lang_users = {}
parsed_langs = parse_multiselect(df["LanguageHaveWorkedWith"])
for i, langs in parsed_langs.items():
    for l in langs:
        lang_users.setdefault(l, []).append(i)

for lang, count in top20_langs:
    idx = lang_users[lang]
    sub = df.loc[df.index.isin(idx), "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Developers who use {lang} report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_language", len(sub))
    add_fact(f"{count/lang_total*100:.0f}% of developers used {lang} in the past year, based on {lang_total} respondents.",
              "adoption_by_language", lang_total)

print(f"Facts after language section: {len(facts)}")

<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18872\1968732752.py:13: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"Developers who use {lang} report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",


Facts after language section: 680


## Career-change facts

In [9]:
for role in devtype_allowlist:
    r = clean_role(role)
    sub = df.loc[df["DevType"] == role, "NewRole"].dropna()
    if len(sub) >= MIN_N:
        considering_pct = sub.str.contains("considered", case=False, na=False).mean() * 100
        add_fact(f"Among {r} respondents, {considering_pct:.0f}% have considered a career or industry change in the past year, based on {len(sub)} respondents.",
                  "career_change_by_role", len(sub))

print(f"Total facts: {len(facts)}")
print(pd.Series([f['category'] for f in facts]).value_counts())

Total facts: 709
tech_by_role                                    258
salary_by_role_country                          102
language_by_country                              45
ai_threat_by_role                                29
ai_trust_by_role                                 29
career_change_by_role                            29
ai_agent_usage_by_role                           29
salary_by_role                                   29
satisfaction_by_role                             28
salary_by_language                               20
adoption_by_language                             20
salary_by_country                                15
salary_by_org_size                                8
salary_by_education                               7
satisfaction_by_education                         7
salary_by_remote                                  5
salary_by_experience                              5
tech_adoption_overall_LanguageHaveWorkedWith      5
tech_adoption_overall_DatabaseHaveWorkedWith   

## Save the facts table

This file is the direct input to Notebook 4 — every row becomes one embedded, searchable fact. **If you're re-running this after Notebook 4 already built an index once, you'll need to re-run Notebook 4 too** — the index has to be rebuilt any time facts.csv changes underneath it.

In [10]:
facts_df = pd.DataFrame(facts)
facts_df.to_csv(f"{DATA_DIR}/facts.csv", index=False)
print(f"Saved {len(facts_df)} facts to {DATA_DIR}/facts.csv")
facts_df.sample(5, random_state=1)[["text", "n"]]

Saved 709 facts to data/facts.csv


,text,n
189,46% of developers used React as their web fram...,22992
111,"In France, the median annual salary among DevO...",41
551,"Among Architect, software or solutions respond...",2304
429,"Among Data or business analyst respondents, 40...",127
666,Developers who use Rust report a median annual...,3036


## What we built

- **709 facts** across 26 categories — covering salary (role, role×country, experience, country, education, management track, company size, remote status, and now specific language), technology adoption (overall, by role, by country, by language), job satisfaction, and five AI-perception cuts
- Every fact still enforces the 30-respondent minimum and states its N explicitly
- Role names read naturally (`"full-stack developer"` instead of the raw `"Developer, full-stack"`), and nothing pluralizes incorrectly regardless of the category name's shape
- The salary-by-language section specifically closes the gap where a real but less-common language (Rust, Go, Kotlin, Swift, ...) could have zero coverage just because it never ranked in a role's top 3

**Next:** re-run Notebook 4 against this new `facts.csv`, then continue to Notebook 5.